# 01 – Data Audit

**Question:** using only information available before a call is made, will a contacted client subscribe to a term deposit (`y` = yes/no)?

**Dataset:** UCI Bank Marketing, `bank-additional-full.csv` (Moro, Rita & Cortez, 2014, CC BY 4.0 – see `data/README.md`).

## 1. Load the data

In [ ]:
#load data
#file uses ";" as separator
import pandas as pd

df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Shape and variable types

In [ ]:
#df.shape and df.dtypes.
print("Rows, columns:", df.shape)
df.dtypes

Rows, columns: (41188, 21)


age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

41,188 rows and 21 columns: 20 inputs and the target `y`. 10 inputs are numeric (`int64`/`float64`) and 10 are categorical (`str`).

## 3. Missing values

In [3]:
df.isna().sum().sum()

np.int64(0)

In [ ]:
#this dataset has no empty cells -> missing answers are written as "unknown" instead. -> count "unknown" in each column.
(df == "unknown").sum()

age                  0
job                330
marital             80
education         1731
default           8597
housing            990
loan               990
contact              0
month                0
day_of_week          0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp.var.rate         0
cons.price.idx       0
cons.conf.idx        0
euribor3m            0
nr.employed          0
y                    0
dtype: int64

There are no empty (NaN) cells, but six categorical columns contain `"unknown"`, mostly `default` (8,597 rows) and `education` (1,731).

Keep `"unknown"` as its own category. Removing these rows would remove about a quarter of the data.

## 4. Duplicates

In [ ]:
df.duplicated().sum()

np.int64(12)

12 rows are exact copies of another row -> dropped, so that the same row doesn't end up in both the training and the test set.

## 5. Target distribution

In [ ]:
# normalize=True gives proportions instead of counts.
print(df["y"].value_counts())
df["y"].value_counts(normalize=True)

y
no     36548
yes     4640
Name: count, dtype: int64


y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

Only **11%** of clients subscribed, so the classes are imbalanced. A model that always answers "no" would already be about 89% accurate -> accuracy alone is not a good metric in this case

## 6. Obvious outliers

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,41188.0,40.024060,10.421250,17.000,32.000,38.000,47.000,98.000
duration,41188.0,258.285010,259.279249,0.000,102.000,180.000,319.000,4918.000
campaign,41188.0,2.567593,2.770014,1.000,1.000,2.000,3.000,56.000
pdays,41188.0,962.475454,186.910907,0.000,999.000,999.000,999.000,999.000
previous,41188.0,0.172963,0.494901,0.000,0.000,0.000,0.000,7.000
emp.var.rate,41188.0,0.081886,1.570960,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,41188.0,93.575664,0.578840,92.201,93.075,93.749,93.994,94.767
cons.conf.idx,41188.0,-40.502600,4.628198,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,41188.0,3.621291,1.734447,0.634,1.344,4.857,4.961,5.045
nr.employed,41188.0,5167.035911,72.251528,4963.600,5099.100,5191.000,5228.100,5228.100


- age from 17 to 98 -> realistic, so the values are kept.
- campaign (number of calls in this campaign) long tail: up to 56 calls & a median of 2. These are real cases -> they are kept.
- pdays is 999 for most rows. This is not a real number of days but a code meaning "not contacted before" (see `data/raw/bank-additional-names.txt`). 
- duration (call length) is only known after the call, when the outcome is already decided -> removed to avoid data leakage

## 7. Summary

| Check | Result | Action |
|---|---|---|
| Shape | 41,188 rows × 21 columns | – |
| Types | 10 numeric, 10 categorical inputs, target `y` | – |
| Missing values | no NaN; `"unknown"` in 6 categorical columns | keep as its own category |
| Duplicates | 12 | drop |
| Target | about 11% yes, 89% no (imbalanced) | stratified split, F1 as main metric |
| Outliers | none impossible; long tail in `campaign`; `pdays` = 999 code | keep |
| Leakage | `duration` known only after the call | drop |

The cleaning (drop duplicates and `duration`) is applied at the start of `02_baseline.ipynb`.